## **비정형 문서 로더 비교**: Unstructured vs Docling vs LlamaParse

- 현대의 AI 애플리케이션에서 PDF, DOCX, PPTX 등의 **복잡한 문서를 정확하게 파싱**하는 것은 매우 중요
- 특히 RAG(Retrieval-Augmented Generation) 시스템에서는 **문서의 구조와 내용을 정확히 이해하고 추출**하는 능력이 성능을 크게 좌우

<br/>

- **종합 비교표**

   | 항목 | Unstructured | Docling | LlamaParse |
   |------|-------------|---------|------------|
   | **라이선스** | 오픈소스 + 상용 | MIT (완전 무료) | 상용 (무료 플랜) |
   | **파일 형식** | 64+ 형식 | 주요 형식 지원 | 10+ 형식 |
   | **처리 속도** | 느림 (50p: 141초) | 중간 (선형 확장) | 매우 빠름 (6초) |
   | **정확도** | 75-100% | 97.9% | 중간 |
   | **테이블 추출** | 단순: 100%, 복잡: 75% | 97.9% | 개선 필요 |
   | **OCR 지원** | ✅ | ✅ | ✅ |
   | **로컬 실행** | ✅ | ✅ | ❌ (API만) |
   | **다국어 지원** | ✅ | ✅ | ✅ |
   | **LangChain 통합** | ✅ | ✅ | ✅ |
   | **LlamaIndex 통합** | ✅ | ✅ | ✅ 네이티브 |

<br/>

- 🏆 **최고 정확도가 필요한 경우 → Docling**
   - 비즈니스 크리티컬한 금융 보고서 분석
   - 복잡한 테이블이 포함된 학술 논문 처리
   - 법률 문서의 정확한 구조 분석
   - 링크: https://github.com/docling-project/docling

- ⚡ **최고 속도가 필요한 경우 → LlamaParse**
   - 실시간 문서 처리 시스템
   - 대량의 문서를 빠르게 처리하는 배치 작업
   - 프로토타이핑 및 초기 개발 단계
   - 링크: https://www.llamaindex.ai/llamaparse

- 🔧 **최고 유연성이 필요한 경우 → Unstructured**
   - 다양한 문서 형식을 처리하는 통합 시스템
   - 복잡한 ETL 파이프라인 구축
   - 엔터프라이즈급 데이터 처리 플랫폼
   - 링크: https://github.com/Unstructured-IO/unstructured

<br/>

- **[출처]**: [PDF Data Extraction Benchmark 2025: Comparing Docling, Unstructured, and LlamaParse for Document Processing Pipelines](https://procycons.com/en/blogs/pdf-data-extraction-benchmark/)

---
## **환경 설정 및 준비**
`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [2]:
import os
from glob import glob
from pathlib import Path

from pprint import pprint
import json

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

`(3) Langsmith tracing 설정`

In [3]:
# Langsmith tracing 여부를 확인 (true: langsmith 추적 활성화, false: langsmith 추적 비활성화)
import os
print(os.getenv('LANGSMITH_TRACING'))

false


---

## **Docling을 활용한 PDF 문서 처리 단계별 가이드**

- Docling은 PDF, DOCX, HTML 등 다양한 문서 형식을 구조화된 데이터로 변환하는 라이브러리
-  PDF 문서 처리를 중심으로 단계별로 학습


- **Docling 설치**

   ```bash
   # pip 설치
   pip install docling
   ```

   ```bash
   # uv 설치
   uv add docling
   ```

### **1. 기본 PDF 변환**

In [4]:
from docling.document_converter import DocumentConverter

# DocumentConverter 초기화 (기본 설정)
converter = DocumentConverter()

# PDF 파일 변환
pdf_path = "data/labor_law.pdf"  # 근로기준법 문서 경로

try:
    # 변환 실행
    result = converter.convert(pdf_path)
    
    # 변환 결과 확인
    if result.status.name in ['SUCCESS', 'PARTIAL_SUCCESS']:
        print(f"변환 성공: {result.status.name}")
        print(f"문서 제목: {result.document.name}")

    else:
        print(f"변환 실패: {result.status}")
        
except Exception as e:
    print(f"오류 발생: {e}")

2025-11-12 23:19:44,264 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-12 23:19:44,297 - INFO - Going to convert document batch...
2025-11-12 23:19:44,298 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-12 23:19:44,330 - INFO - Loading plugin 'docling_defaults'
2025-11-12 23:19:44,333 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-12 23:19:44,371 - INFO - Loading plugin 'docling_defaults'
2025-11-12 23:19:44,371 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-12 23:19:44,632 - INFO - Accelerator device: 'cpu'
[INFO] 2025-11-12 23:19:44,651 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-11-12 23:19:44,659 [RapidOCR] download_file.py:60: File exists and is valid: H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-11-12 23:19:44,659 [RapidOCR] main.py:53: Using H:\mi

변환 성공: SUCCESS
문서 제목: labor_law


In [5]:
type(result)  # ConversionResult 타입 확인

docling.datamodel.document.ConversionResult

In [6]:
result.model_dump().keys()  # ConversionResult의 속성 키 확인

dict_keys(['input', 'status', 'errors', 'pages', 'assembled', 'timings', 'confidence', 'document'])

### **2. 텍스트 추출 및 마크다운 변환**

In [7]:
# DoclingDocument 객체로 변환
document = result.document

In [8]:
document.model_dump()  # DoclingDocument의 속성 확인

{'schema_name': 'DoclingDocument',
 'version': '1.8.0',
 'name': 'labor_law',
 'origin': {'mimetype': 'application/pdf',
  'binary_hash': 659731770781194427,
  'filename': 'labor_law.pdf',
  'uri': None},
 'furniture': {'self_ref': '#/furniture',
  'parent': None,
  'children': [],
  'content_layer': <ContentLayer.FURNITURE: 'furniture'>,
  'meta': None,
  'name': '_root_',
  'label': <GroupLabel.UNSPECIFIED: 'unspecified'>},
 'body': {'self_ref': '#/body',
  'parent': None,
  'children': [{'cref': '#/texts/0'},
   {'cref': '#/texts/1'},
   {'cref': '#/groups/0'},
   {'cref': '#/texts/3'},
   {'cref': '#/groups/1'},
   {'cref': '#/texts/19'},
   {'cref': '#/texts/20'},
   {'cref': '#/texts/21'},
   {'cref': '#/groups/2'},
   {'cref': '#/pictures/0'},
   {'cref': '#/texts/30'},
   {'cref': '#/texts/31'},
   {'cref': '#/texts/32'},
   {'cref': '#/texts/33'},
   {'cref': '#/texts/34'},
   {'cref': '#/texts/35'},
   {'cref': '#/texts/36'},
   {'cref': '#/texts/37'},
   {'cref': '#/texts/38

In [9]:
# 마크다운으로 내보내기
markdown_text = document.export_to_markdown()

print(markdown_text[:500] + "...")  # 처음 500자만 출력

## 제1장 총칙

- 제1조(목적) 이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.

제2조(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. &lt;개정 2018. 3. 20., 2019. 1. 15., 2020. 5. 26.&gt;

1. '근로자'란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. '사용자'란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대하여 사업주를 위하여 행위하는 자를 말한다.
3. '근로'란 정신노동과 육체노동을 말한다.
4. '근로계약'이란 근로자가 사용자에게 근로를 제공하고 사용자는 이에 대하여 임금을 지급하는 것을 목적으로 체 결된 계약을 말한다.
5. '임금'이란 사용자가 근로의 대가로 근로자에게 임금, 봉급, 그 밖에 어떠한 명칭으로든지 지급하는 모든 금품을 말한다.
6. '...


In [10]:
# 순수 텍스트로 내보내기
plain_text = document.export_to_text()
print(f"{len(plain_text)}자")

print(plain_text[:300] + "...")  # 처음 300자만 출력

2025-11-12 23:20:02,859 - WARNING - Parameter `strict_text` has been deprecated and will be ignored.


40407자
## 제1장 총칙

- 제1조(목적) 이 법은 헌법에 따라 근로조건의 기준을 정함으로써 근로자의 기본적 생활을 보장, 향상시키며 균형 있는 국민경제의 발전을 꾀하는 것을 목적으로 한다.

제2조(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. &lt;개정 2018. 3. 20., 2019. 1. 15., 2020. 5. 26.&gt;

1. '근로자'란 직업의 종류와 관계없이 임금을 목적으로 사업이나 사업장에 근로를 제공하는 사람을 말한다.
2. '사용자'란 사업주 또는 사업 경영 담당자, 그 밖에 근로자에 관한 사항에 대...


In [11]:
# 파일로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "labor_law.txt", "w", encoding="utf-8") as f:
    f.write(plain_text)

### **3. 고급 설정 (OCR, 테이블 구조)**

In [12]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.accelerator_options import AcceleratorDevice

class AdvancedDocProcessor:
    """고급 문서 처리기"""
    
    def __init__(self, enable_ocr=False, enable_table_structure=True):
        """
        Args:
            enable_ocr: OCR 기능 활성화 (스캔된 문서용)
            enable_table_structure: 테이블 구조 분석 활성화
        """
        
        # 파이프라인 옵션 설정
        pipeline_options = PdfPipelineOptions()  # PDF 변환을 위한 파이프라인 옵션 (기본값 사용)
        pipeline_options.do_ocr = enable_ocr   # OCR 활성화 여부
        pipeline_options.do_table_structure = enable_table_structure # 테이블 구조 분석 활성화 여부 
        
        # 테이블 구조 분석 세부 설정
        if enable_table_structure:
            pipeline_options.table_structure_options.do_cell_matching = True # 셀 매칭
        
        # CPU 사용 설정 (GPU가 없는 환경)
        pipeline_options.accelerator_options.device = AcceleratorDevice.CPU
        
        # DocumentConverter 초기화
        self.converter = DocumentConverter(
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline_options
                )
            }
        )
        
        print(f"🔧 처리기 초기화 완료:")
        print(f"   - OCR: {'활성화' if enable_ocr else '비활성화'}")
        print(f"   - 테이블 구조 분석: {'활성화' if enable_table_structure else '비활성화'}")
    
    def process_pdf(self, pdf_path):
        """PDF 처리"""
        try:
            print(f"처리 시작: {pdf_path}")
            result = self.converter.convert(str(pdf_path))
            
            if result.status.name in ['SUCCESS', 'PARTIAL_SUCCESS']:
                print(f"처리 완료: {result.status.name}")
                return result.document
            else:
                print(f"처리 실패: {result.status}")
                return None
                
        except Exception as e:
            print(f"오류 발생: {e}")
            return None

`(1) 텍스트만 추출`

In [13]:
# 기본 설정으로 처리
basic_processor = AdvancedDocProcessor(
    enable_ocr=False,
    enable_table_structure=False
)

# PDF 처리 실행
pdf_path = "data/transformer.pdf"  # 변환할 PDF 문서 경로 
document = basic_processor.process_pdf(pdf_path)

2025-11-12 23:20:02,889 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-12 23:20:02,889 - INFO - Going to convert document batch...
2025-11-12 23:20:02,889 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 34740f9bc344b818f5f4dd4f4b67a9b3
2025-11-12 23:20:02,889 - INFO - Accelerator device: 'cpu'


🔧 처리기 초기화 완료:
   - OCR: 비활성화
   - 테이블 구조 분석: 비활성화
처리 시작: data/transformer.pdf


2025-11-12 23:20:03,502 - INFO - Processing document transformer.pdf
2025-11-12 23:20:20,211 - INFO - Finished converting document transformer.pdf in 17.31 sec.


처리 완료: SUCCESS


In [14]:
# 결과 분석
markdown = document.export_to_markdown()

print(f"- 문서명: {document.name}")
print(f"- 텍스트 길이: {len(markdown)}자")

# 테이블이 있는지 확인
if "| " in markdown or "|--" in markdown:
    print("   - 🔍 테이블 구조 감지됨")
else:
    print("   - 📝 일반 텍스트 문서")

- 문서명: transformer
- 텍스트 길이: 41233자
   - 🔍 테이블 구조 감지됨


In [15]:
# 마크다운 파일로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "transformer_analysis.md", "w", encoding="utf-8") as f:
    f.write(markdown)

`(2) 테이블 구조 추출`

- **테슬라 10-K 보고서** PDF 파일을 **인터넷에서 다운로드**

- **SEC EDGAR 웹사이트** 또는 **테슬라 IR 페이지**에서 다운로드 가능

- 출처: https://ir.tesla.com/#quarterly-disclosure

In [16]:
# 고급 설정으로 처리
ocr_processor = AdvancedDocProcessor(
    enable_ocr=True,  # OCR 활성화
    enable_table_structure=True # 테이블 구조 분석 활성화
)

# PDF 처리 실행
pdf_path = "data/tsla-20241231-gen.pdf"  # 변환할 PDF 문서 경로 
document = ocr_processor.process_pdf(pdf_path)

2025-11-12 23:20:20,323 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-12 23:20:20,327 - INFO - Going to convert document batch...
2025-11-12 23:20:20,328 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 79859c21383f9cceb3a0700faabde318
2025-11-12 23:20:20,329 - INFO - Accelerator device: 'cpu'
[INFO] 2025-11-12 23:20:20,347 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-11-12 23:20:20,352 [RapidOCR] download_file.py:60: File exists and is valid: H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-11-12 23:20:20,352 [RapidOCR] main.py:53: Using H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2025-11-12 23:20:20,437 [RapidOCR] base.py:22: Using engine_name: onnxruntime


🔧 처리기 초기화 완료:
   - OCR: 활성화
   - 테이블 구조 분석: 활성화
처리 시작: data/tsla-20241231-gen.pdf


[INFO] 2025-11-12 23:20:20,439 [RapidOCR] download_file.py:60: File exists and is valid: H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-11-12 23:20:20,440 [RapidOCR] main.py:53: Using H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2025-11-12 23:20:20,483 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2025-11-12 23:20:20,492 [RapidOCR] download_file.py:60: File exists and is valid: H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
[INFO] 2025-11-12 23:20:20,492 [RapidOCR] main.py:53: Using H:\miniconda3\envs\modu3\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_rec_infer.onnx
2025-11-12 23:20:20,595 - INFO - Auto OCR model selected rapidocr with onnxruntime.
2025-11-12 23:20:20,596 - INFO - Accelerator device: 'cpu'
2025-11-12 23:20:21,424 - INFO - Accelerator device: 'cpu'
2025-11-12 23:20:21,704 - INFO - Processing document 

처리 완료: SUCCESS


In [17]:
# 결과 분석
markdown = document.export_to_markdown()

print(f"- 문서명: {document.name}")
print(f"- 텍스트 길이: {len(markdown)}자")

# 테이블이 있는지 확인
if "| " in markdown or "|--" in markdown:
    print("   - 🔍 테이블 구조 감지됨")
else:
    print("   - 📝 일반 텍스트 문서")

- 문서명: tsla-20241231-gen
- 텍스트 길이: 610830자
   - 🔍 테이블 구조 감지됨


In [18]:
# 마크다운 파일로 저장
save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

with open(save_folder / "tsla_analysis_ocr.md", "w", encoding="utf-8") as f:
    f.write(markdown)

### **4. 문서 구조 분석**

In [19]:
# DoclingDocument의 속성 확인
document.model_dump().keys()  

dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])

In [20]:
# DoclingDocument의 텍스트 내용 확인
document.texts

[TextItem(self_ref='#/texts/0', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=14.36, t=668.761, r=61.752, b=660.041, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 10))], orig='(Mark\tOne)', text='(Mark\tOne)', formatting=None, hyperlink=None),
 ListItem(self_ref='#/texts/1', parent=RefItem(cref='#/groups/0'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.LIST_ITEM: 'list_item'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=15.126, t=656.083, r=455.215, b=647.362, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 86))], orig='x ANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934', text='x ANNUAL\tREPORT\tPURSUANT\tTO\tSECTION\t13\tOR\t15(d)\tOF\tTHE\tSECURITIES\tEXCHANGE\tACT\tOF\t1934', formatting=None, hyperlink=None, e

In [21]:
# 첫 번째 텍스트 아이템의 속성 확인
document.texts[0].model_dump()  

{'self_ref': '#/texts/0',
 'parent': {'cref': '#/body'},
 'children': [],
 'content_layer': <ContentLayer.BODY: 'body'>,
 'meta': None,
 'label': <DocItemLabel.TEXT: 'text'>,
 'prov': [{'page_no': 1,
   'bbox': {'l': 14.36,
    't': 668.761,
    'r': 61.752,
    'b': 660.041,
    'coord_origin': <CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>},
   'charspan': (0, 10)}],
 'orig': '(Mark\tOne)',
 'text': '(Mark\tOne)',
 'formatting': None,
 'hyperlink': None}

In [22]:
# DoclingDocument의 테이블 구조 확인
document.tables

[TableItem(self_ref='#/tables/0', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TABLE: 'table'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=13.474660873413086, t=388.309326171875, r=597.6355590820312, b=360.18450927734375, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 0))], captions=[], references=[], footnotes=[], image=None, data=TableData(table_cells=[TableCell(bbox=BoundingBox(l=80.714, t=407.88, r=142.791, b=415.93, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=0, end_col_offset_idx=1, text='Title of each class', column_header=True, row_header=False, row_section=False, fillable=False), TableCell(bbox=BoundingBox(l=274.326, t=407.88, r=337.604, b=415.93, coord_origin=<CoordOrigin.TOPLEFT: 'TOPLEFT'>), row_span=1, col_span=1, start_row_offset_idx=0, end_row_offset_idx=1, start_col_offset_idx=

In [23]:
# 첫 번째 테이블 아이템의 속성 확인
document.tables[0].model_dump() 

{'self_ref': '#/tables/0',
 'parent': {'cref': '#/body'},
 'children': [],
 'content_layer': <ContentLayer.BODY: 'body'>,
 'meta': None,
 'label': <DocItemLabel.TABLE: 'table'>,
 'prov': [{'page_no': 1,
   'bbox': {'l': 13.474660873413086,
    't': 388.309326171875,
    'r': 597.6355590820312,
    'b': 360.18450927734375,
    'coord_origin': <CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>},
   'charspan': (0, 0)}],
 'captions': [],
 'references': [],
 'footnotes': [],
 'image': None,
 'data': {'table_cells': [{'bbox': {'l': 80.714,
     't': 407.88,
     'r': 142.791,
     'b': 415.93,
     'coord_origin': <CoordOrigin.TOPLEFT: 'TOPLEFT'>},
    'row_span': 1,
    'col_span': 1,
    'start_row_offset_idx': 0,
    'end_row_offset_idx': 1,
    'start_col_offset_idx': 0,
    'end_col_offset_idx': 1,
    'text': 'Title of each class',
    'column_header': True,
    'row_header': False,
    'row_section': False,
    'fillable': False},
   {'bbox': {'l': 274.326,
     't': 407.88,
     'r': 337.604,
 

In [24]:
# DoclingDocument의 텍스트와 테이블 요소를 순서대로 JSON으로 변환
from docling_core.types.doc import TextItem, TableItem
import pandas as pd

def convert_document_to_ordered_json(document):
    """문서 요소를 원본 순서대로 JSON 배열로 변환 (테이블은 마크다운+딕셔너리 형식)"""
    elements = []
    
    for item, level in document.iterate_items():
        if isinstance(item, TextItem):
            elements.append({
                "type": "text",
                "content": item.text.replace("\t", " ") or "",
                "page": item.prov[0].page_no if item.prov else None,
                "label": item.label.value if item.label else None,
                "level": level,
                "element": item  # 원본 요소 추가
            })
        elif isinstance(item, TableItem):
            table_content = {}
            
            try:
                # DataFrame으로 변환
                df = item.export_to_dataframe()

                # 마크다운 형식으로 변환
                markdown_content = df.to_markdown(index=False)  # 인덱스 제외

                # 딕셔너리 형식으로 변환 (여러 옵션 제공)
                dict_content = df.to_dict('records')     # 각 행을 딕셔너리로

                table_content = {
                    "markdown": markdown_content,
                    "data": dict_content,
                    "status": "success"
                }
                
            except Exception as e:
                # DataFrame 변환이 실패한 경우 대안 시도
                try:
                    html_content = item.export_to_html()
                    table_content = {
                        "html": html_content,
                        "status": "html_fallback",
                        "error": str(e)
                    }
                except Exception as e2:
                    table_content = {
                        "status": "failed",
                        "error": f"DataFrame 변환 실패: {str(e)}, HTML 변환 실패: {str(e2)}"
                    }
            
            elements.append({
                "type": "table",
                "content": table_content,
                "page": item.prov[0].page_no if item.prov else None,
                "level": level,
                "element": item  # 원본 요소 추가
            })
    
    return elements

# 문서 요소를 원본 순서대로 JSON 배열로 변환
ordered_json = convert_document_to_ordered_json(document)
print(f"문서 요소를 원본 순서대로 JSON 배열로 변환 완료. 총 {len(ordered_json)}개 요소.")

2025-11-12 23:25:11,670 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,675 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,686 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,690 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,693 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,697 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,704 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,707 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:11,707 - WARNING - Usage of TableItem.export_to_dataframe() wit

문서 요소를 원본 순서대로 JSON 배열로 변환 완료. 총 1538개 요소.


In [25]:
ordered_json[:10]

[{'type': 'text',
  'content': '(Mark One)',
  'page': 1,
  'label': 'text',
  'level': 1,
  'element': TextItem(self_ref='#/texts/0', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TEXT: 'text'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=14.36, t=668.761, r=61.752, b=660.041, coord_origin=<CoordOrigin.BOTTOMLEFT: 'BOTTOMLEFT'>), charspan=(0, 10))], orig='(Mark\tOne)', text='(Mark\tOne)', formatting=None, hyperlink=None)},
 {'type': 'text',
  'content': 'x ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934',
  'page': 1,
  'label': 'list_item',
  'level': 2,
  'element': ListItem(self_ref='#/texts/1', parent=RefItem(cref='#/groups/0'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.LIST_ITEM: 'list_item'>, prov=[ProvenanceItem(page_no=1, bbox=BoundingBox(l=15.126, t=656.083, r=455.215, b=647.362, coord_origin=<CoordOrigin.BOTTOMLEFT: 

In [26]:
# 표 객체 확인
table_indices = []
for i, item in enumerate(ordered_json):
    if item['type'] == 'table':
        table_indices.append(i)

print(len(table_indices))
print(table_indices)


83
[15, 48, 362, 395, 428, 444, 461, 465, 469, 472, 475, 479, 497, 514, 540, 544, 548, 552, 556, 572, 580, 637, 639, 641, 647, 680, 697, 722, 725, 727, 729, 730, 738, 741, 742, 748, 753, 758, 763, 766, 768, 789, 797, 799, 802, 804, 806, 810, 814, 819, 826, 834, 838, 840, 842, 844, 851, 896, 902, 906, 908, 910, 949, 951, 952, 953, 954, 956, 957, 958, 959, 960, 962, 963, 964, 965, 981, 1316, 1317, 1478, 1485, 1486, 1487]


In [27]:
# 첫 번째 테이블 요소 확인
if len(table_indices) > 0:
    first_table_idx = table_indices[0]
    item = ordered_json[first_table_idx]
    
    print(f"첫 번째 테이블:")
    print(f"  - 인덱스: {first_table_idx}")
    print(f"  - 페이지: {item.get('page', 'N/A')}")
    print(f"  - 상태: {item['content']['status']}")
    
    if item['content']['status'] == 'success':
        print(f"\n[마크다운 형식 미리보기]")
        markdown = item['content']['markdown']
        lines = markdown.split('\n')
        preview_lines = lines[:10]  # 처음 10줄만
        print('\n'.join(preview_lines))
        if len(lines) > 10:
            print(f"... (총 {len(lines)}줄 중 10줄만 표시)")
else:
    print("테이블이 없습니다.")

첫 번째 테이블:
  - 인덱스: 15
  - 페이지: 1
  - 상태: success

[마크다운 형식 미리보기]
| Title of each class   | Trading Symbol(s)   | Name of each exchange on which registered   |
|:----------------------|:--------------------|:--------------------------------------------|
| Common stock          | TSLA                | The Nasdaq Global Select Market             |


In [28]:
# 테이블 요소의 내용 확인
# 먼저 테이블 인덱스를 확인 (이전 셀에서 table_indices 실행 필요)

# 안전하게 접근하기
if 397 < len(ordered_json):
    item = ordered_json[397]
    print(f"요소 타입: {item['type']}")
    print(f"페이지: {item.get('page', 'N/A')}")
    
    if item['type'] == 'table' and isinstance(item['content'], dict):
        # 테이블인 경우
        if item['content']['status'] == 'success':
            print("\n[마크다운 형식]")
            print(item['content']['markdown'])
        else:
            print(f"\n오류: {item['content'].get('error', 'Unknown error')}")
    elif item['type'] == 'text':
        # 텍스트인 경우
        print(f"\n[텍스트 내용]")
        print(item['content'][:200] + "..." if len(item['content']) > 200 else item['content'])
else:
    print(f"인덱스 397은 범위를 벗어났습니다. (총 요소 수: {len(ordered_json)})")

요소 타입: text
페이지: 36

[텍스트 내용]
These plans are subject to uncertainties inherent in establishing and ramping manufacturing operations, which may be exacerbated by new product and manufacturing technologies we introduce, the number ...


In [29]:
# 테이블 요소를 DataFrame으로 확인

if 397 < len(ordered_json):
    item = ordered_json[397]
    
    if item['type'] == 'table' and isinstance(item['content'], dict):
        # 테이블인 경우
        if item['content']['status'] == 'success':
            # DataFrame으로 변환하여 표시
            df = pd.DataFrame(item['content']['data'])
            print(f"테이블 크기: {df.shape[0]} rows × {df.shape[1]} columns")
            print("\n[DataFrame 형식]")
            display(df)
        else:
            print(f"테이블 변환 실패: {item['content'].get('error', 'Unknown error')}")
    else:
        print(f"인덱스 397은 테이블이 아닙니다. (타입: {item['type']})")
        print("테이블 인덱스를 확인하려면 이전 셀(표 객체 확인)을 실행하세요.")
else:
    print(f"인덱스 397은 범위를 벗어났습니다. (총 요소 수: {len(ordered_json)})")

인덱스 397은 테이블이 아닙니다. (타입: text)
테이블 인덱스를 확인하려면 이전 셀(표 객체 확인)을 실행하세요.


In [30]:
# pickle로 저장
import pickle

save_folder = Path("data/docling_output")
pickle_path = save_folder / "tsla_analysis_ocr.pkl"

with open(pickle_path, "wb") as f:
    pickle.dump(ordered_json, f)

print(f"pickle 파일로 저장됨: {pickle_path}")

pickle 파일로 저장됨: data\docling_output\tsla_analysis_ocr.pkl


# [실습] 

- transformer.pdf 파일 처리
- 테이블 요소와 텍스트 요소를 각각 구분하여 정리
    - 테이블 요소는 마크다운 형식으로 저장
    - 텍스트 요소는 텍스트 형식으로 저장

In [31]:
# [실습] transformer.pdf 파일 처리

# 1단계: PDF 파일 처리 (OCR 비활성화, 테이블 구조 분석 활성화)
print("=" * 60)
print("1단계: PDF 파일 처리")
print("=" * 60)

# 기본 설정으로 처리기 초기화
processor = AdvancedDocProcessor(
    enable_ocr=False,
    enable_table_structure=True
)

# transformer.pdf 파일 처리
pdf_path = "data/transformer.pdf"
document = processor.process_pdf(pdf_path)

if document is None:
    print("❌ 문서 처리 실패")
else:
    print(f"✅ 문서 처리 완료: {document.name}")

print("\n" + "=" * 60)
print("2단계: 문서 요소 추출")
print("=" * 60)

# 2단계: 문서 요소를 원본 순서대로 JSON 배열로 변환
ordered_json = convert_document_to_ordered_json(document)
print(f"✅ 총 {len(ordered_json)}개 요소 추출 완료")

# 3단계: 테이블과 텍스트 요소 분리
print("\n" + "=" * 60)
print("3단계: 테이블 및 텍스트 요소 분리")
print("=" * 60)

table_elements = []
text_elements = []

for item in ordered_json:
    if item['type'] == 'table':
        table_elements.append(item)
    elif item['type'] == 'text':
        text_elements.append(item)

print(f"✅ 테이블 요소: {len(table_elements)}개")
print(f"✅ 텍스트 요소: {len(text_elements)}개")

# 4단계: 테이블 요소를 마크다운 형식으로 저장
print("\n" + "=" * 60)
print("4단계: 테이블 요소 마크다운 저장")
print("=" * 60)

save_folder = Path("data/docling_output")
save_folder.mkdir(parents=True, exist_ok=True)

tables_md_content = []
for idx, table in enumerate(table_elements):
    tables_md_content.append(f"## 테이블 {idx + 1}\n")
    tables_md_content.append(f"**페이지**: {table['page']}\n\n")
    
    if table['content']['status'] == 'success':
        tables_md_content.append(table['content']['markdown'])
    elif table['content']['status'] == 'html_fallback':
        tables_md_content.append(f"```html\n{table['content']['html']}\n```")
    else:
        tables_md_content.append(f"*오류: {table['content']['error']}*")
    
    tables_md_content.append("\n\n---\n\n")

# 테이블 마크다운 파일 저장
tables_md_path = save_folder / "transformer_tables.md"
with open(tables_md_path, "w", encoding="utf-8") as f:
    f.write("\n".join(tables_md_content))

print(f"✅ 테이블 마크다운 저장: {tables_md_path}")
print(f"   - 총 {len(table_elements)}개 테이블 저장됨")

# 5단계: 텍스트 요소를 텍스트 형식으로 저장
print("\n" + "=" * 60)
print("5단계: 텍스트 요소 텍스트 저장")
print("=" * 60)

text_content = []
for item in text_elements:
    if item['content']:  # 내용이 있는 경우만
        text_content.append(item['content'])

# 텍스트 파일 저장
text_path = save_folder / "transformer_text.txt"
with open(text_path, "w", encoding="utf-8") as f:
    f.write("\n".join(text_content))

print(f"✅ 텍스트 파일 저장: {text_path}")
print(f"   - 총 {len(text_elements)}개 텍스트 요소 중 {len(text_content)}개 저장됨")
print(f"   - 전체 텍스트 길이: {len(''.join(text_content))}자")

# 결과 요약
print("\n" + "=" * 60)
print("📊 처리 결과 요약")
print("=" * 60)
print(f"문서명: {document.name}")
print(f"총 요소 수: {len(ordered_json)}개")
print(f"  - 테이블: {len(table_elements)}개")
print(f"  - 텍스트: {len(text_elements)}개")
print(f"\n저장된 파일:")
print(f"  - {tables_md_path}")
print(f"  - {text_path}")
print("=" * 60)

2025-11-12 23:25:12,090 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2025-11-12 23:25:12,090 - INFO - Going to convert document batch...
2025-11-12 23:25:12,090 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 0ac8c49c54a6331ee7852e244622aa0b
2025-11-12 23:25:12,090 - INFO - Accelerator device: 'cpu'


1단계: PDF 파일 처리
🔧 처리기 초기화 완료:
   - OCR: 비활성화
   - 테이블 구조 분석: 활성화
처리 시작: data/transformer.pdf


2025-11-12 23:25:13,222 - INFO - Accelerator device: 'cpu'
2025-11-12 23:25:13,508 - INFO - Processing document transformer.pdf
2025-11-12 23:25:41,703 - INFO - Finished converting document transformer.pdf in 29.62 sec.
2025-11-12 23:25:41,709 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:41,712 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:41,715 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
2025-11-12 23:25:41,724 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


처리 완료: SUCCESS
✅ 문서 처리 완료: transformer

2단계: 문서 요소 추출
✅ 총 171개 요소 추출 완료

3단계: 테이블 및 텍스트 요소 분리
✅ 테이블 요소: 4개
✅ 텍스트 요소: 167개

4단계: 테이블 요소 마크다운 저장
✅ 테이블 마크다운 저장: data\docling_output\transformer_tables.md
   - 총 4개 테이블 저장됨

5단계: 텍스트 요소 텍스트 저장
✅ 텍스트 파일 저장: data\docling_output\transformer_text.txt
   - 총 167개 텍스트 요소 중 162개 저장됨
   - 전체 텍스트 길이: 35599자

📊 처리 결과 요약
문서명: transformer
총 요소 수: 171개
  - 테이블: 4개
  - 텍스트: 167개

저장된 파일:
  - data\docling_output\transformer_tables.md
  - data\docling_output\transformer_text.txt
